In [2]:
import requests
import shutil
import csv
import os
import random
from tqdm import tqdm
import urllib

In [3]:
def download_images(csv_file, image_folder, number_of_images_per_class):
    with open(csv_file, 'r') as file:
        reader = csv.reader(file)
        next(reader)  # Skip header row

        rows = list(reader)
        if number_of_images_per_class > len(rows):
            print(f"Requested {number_of_images_per_class} images, but only {len(rows)} are available. Downloading all available images.")
            number_of_images_per_class = len(rows)
        random_rows = random.sample(rows, number_of_images_per_class)

        for row in tqdm(random_rows):
            image_url = row[1]
            extension = os.path.splitext(urllib.parse.urlparse(image_url).path)[1]
            filename = row[0] + extension
            file_path = os.path.join(image_folder, filename)

            try:
                response = requests.get(image_url, stream=True)
                response.raise_for_status()  # Check if the request was successful

                with open(file_path, 'wb') as out_file:
                    shutil.copyfileobj(response.raw, out_file)

            except requests.exceptions.RequestException as e:
                print(f"Failed to download {image_url}: {e}")

In [4]:
root_dir = "./"
csv_files = os.path.join(root_dir, "csv_files/")
os.makedirs(os.path.join(root_dir, "images/"), exist_ok=True) # Create images directory if it doesn't exist
image_folders = os.path.join(root_dir, "images")

In [5]:
number_of_images_per_class = 350

for each_csv in os.listdir(csv_files):
    folder_name = each_csv.split(".")[0]
    # Create a folder for each class
    os.makedirs(os.path.join(image_folders, folder_name), exist_ok=True)
    csv_file = os.path.join(csv_files, each_csv)
    image_folder = os.path.join(image_folders, folder_name)
    download_images(csv_file, image_folder, number_of_images_per_class)

  0%|          | 0/350 [00:00<?, ?it/s]

100%|██████████| 350/350 [16:35<00:00,  2.84s/it]


In [6]:

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2" #show only error messages, to avoid printing a tonne of verbose warnings!

import tensorflow as tf
tf.get_logger().setLevel('ERROR') #set TensorFlow logger level to ERROR 
print("tensorflow: ",tf.__version__)

import keras
print("keras: ",keras.__version__)

import keras_hub
print("keras hub: ",keras_hub.__version__)

import keras_cv
print("keras cv: ",keras_cv.__version__)

tensorflow:  2.20.0
keras:  3.14.1
keras hub:  0.29.0
keras cv:  0.9.0


In [7]:
print(keras_hub.models.ImageObjectDetector.presets.keys())

dict_keys(['dfine_nano_coco', 'dfine_small_coco', 'dfine_medium_coco', 'dfine_large_coco', 'dfine_xlarge_coco', 'dfine_small_obj365', 'dfine_medium_obj365', 'dfine_large_obj365', 'dfine_xlarge_obj365', 'dfine_small_obj2coco', 'dfine_medium_obj2coco', 'dfine_large_obj2coco_e25', 'dfine_xlarge_obj2coco', 'retinanet_resnet50_fpn_coco', 'retinanet_resnet50_fpn_v2_coco'])


In [8]:
retinanet = keras_hub.models.RetinaNetObjectDetector.from_preset(
    "retinanet_resnet50_fpn_coco"
)

100%|██████████| 1.59k/1.59k [00:00<00:00, 1.38MB/s]


100%|██████████| 8.54k/8.54k [00:00<00:00, 827kB/s]
E0000 00:00:1778334266.490635   34148 cuda_executor.cc:1309] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1778334266.510331   34148 gpu_device.cc:2342] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


100%|██████████| 131M/131M [00:50<00:00, 2.69MB/s] 


100%|██████████| 105M/105M [01:11<00:00, 1.55MB/s] 


In [10]:
image_size=(800, 800)
batch_size=8

image_dir = "./images/"

tf_dataset = keras.utils.image_dataset_from_directory(
    image_dir,
    image_size=image_size,
    batch_size=batch_size,
    pad_to_aspect_ratio=True
)

Found 770 files belonging to 1 classes.


In [ ]:
from keras_cv import visualization

for images, labels in tf_dataset.take(2):
    print(images.shape)

    y_pred = retinanet.predict(images)

    keras_cv_preds = {
        "boxes": y_pred["boxes"],
        "classes": y_pred["labels"],
        "confidence": y_pred["confidence"],
    }
    visualization.plot_bounding_box_gallery(
        images,
        value_range=(0, 255),
        rows=2,
        cols=4,
        y_pred=keras_cv_preds,
        scale=5,
        font_scale=0.7,
        bounding_box_format="yxyx",
        class_mapping=class_mapping,
    )

(8, 800, 800, 3)
1/1 ━━━━━━━━━━━━━━━━━━━━ 235s 235s/step


AttributeError: 'RetinaNetObjectDetector' object has no attribute 'class_mapping'